In [11]:
import pandas as pd
import datetime as dt
import numpy as np

from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.express as px


In [2]:
import os
if os.path.exists('pass.env'):
    load_dotenv('pass.env')
else:
    load_dotenv('../pass.env')

#Environment Variables
DB_TYPE = os.getenv('DB_TYPE')
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

In [3]:
# connect engine
connection_string = f"{DB_TYPE}://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

In [4]:
query = "SELECT * FROM fact_customer_transactions;"
df = pd.read_sql_query(query, con=engine)

In [ ]:
# RFM analysis
df['order_date'] = pd.to_datetime(df['order_date'])

snapshot_date = df['order_date'].max() + dt.timedelta(days=1)

rfm = df.groupby('customer_unique_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,  # Recency
    'order_id': 'nunique',                                   # Frequency
    'total_payment_value': 'sum'                             # Monetary
}).reset_index()

rfm.columns = ['customer_unique_id', 'Recency', 'Frequency', 'Monetary']

print(rfm.head())

print(rfm.describe())

# plot rfm segment map
rfm_log = rfm[['Recency', 'Frequency', 'Monetary']].apply(np.log1p)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary'])

# 2. Train K-Means Model & Assign Clusters

# Create 4 customer segments
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans.fit(rfm_scaled_df)

#  THIS IS THE CRITICAL LINE! 
rfm['Cluster'] = kmeans.labels_

print(" 'Cluster' column successfully created! Preparing 3D Plot...")


# Generate 3D Interactive Scatter Plot (Plotly)

# Convert 'Cluster' to string so Plotly treats it as categories (distinct colors)
rfm['Cluster_Label'] = 'Cluster ' + rfm['Cluster'].astype(str)

# Plot R, F, M on X, Y, Z axes
fig = px.scatter_3d(
    rfm,
    x='Recency',
    y='Frequency',
    z='Monetary',
    color='Cluster_Label',     # Color points by our newly created Cluster column
    opacity=0.7,
    title='3D RFM Customer Segmentation Map',
    labels={
        'Recency': 'Recency',
        'Frequency': 'Frequency',
        'Monetary': 'Monetary'
    },
    color_discrete_sequence=px.colors.qualitative.Set1
)

# Refine marker size for better performance and aesthetics
fig.update_traces(marker=dict(size=4, line=dict(width=0)))

# Render the interactive visualization
fig.show()

                 customer_unique_id  Recency  Frequency  Monetary
0  0000366f3b9a7992bf8c76cfdf3221e2      161          1    141.90
1  0000b849f77a49e4a4ce2b2a4ca5be3f      164          1     27.19
2  0000f46a3911fa3c0805444483337064      587          1     86.22
3  0000f6ccb0745a6a4b88665a16c9f078      371          1     43.62
4  0004aac84e0df4da2b147fca70cf8255      338          1    196.89
            Recency     Frequency      Monetary
count  96096.000000  96096.000000  96096.000000
mean     289.108797      1.034809    166.592492
std      153.417869      0.214384    231.428332
min        1.000000      1.000000      0.000000
25%      165.000000      1.000000     63.120000
50%      270.000000      1.000000    108.000000
75%      398.000000      1.000000    183.530000
max      774.000000     17.000000  13664.080000
✅ 'Cluster' column successfully created! Preparing 3D Plot...


In [ ]:
rfm_log = rfm[['Recency', 'Frequency', 'Monetary']].apply(np.log1p)

# Step 2: Feature Scaling (Standardization)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

# Convert the scaled array back to a DataFrame for easier handling
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary'])


# Train the K-Means Clustering Model

# We select k=4 representing general business segments: VIP, Loyal, New, At Risk.
# random_state=42 ensures reproducibility of the clusters.
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

kmeans.fit(rfm_scaled_df)

# Assign the predicted cluster labels (0, 1, 2, 3) back to the original dataframe
rfm['Cluster'] = kmeans.labels_


# Cluster Profiling (Summary for Business Interpretation)


cluster_summary = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'count']
}).round(2)

print(cluster_summary)

✅ Preprocessing complete! Data is ready for modeling.
🧠 AI is currently assigning clusters to customers...
✅ Customer segmentation complete!

📊 Cluster Summary (for business interpretation):
        Recency Frequency Monetary       
           mean      mean     mean  count
Cluster                                  
0        117.33      1.00   122.08  25503
1        335.94      1.00   322.36  28659
2        269.57      2.12   314.99   2997
3        368.66      1.00    69.67  38937
